# SQL

- [Перейти к подзапросы и CTE (WITH)](#cte-section)
- [Перейти к оконным функциям](#over)
- [Перейти к транзакции в SQL и ACID](#acid)

*SQL здесь рассматривается в рамках библиотек SQLAlchemy и Pandas. Логично, что на практике sql не ограничевается таким способом!!!*  
*Все реальные запросы sql будут внутри функции text() и без """_"""*

*Ещё важный нюанс! Если sql пишется напрямую в базе, то запросы следует заканчивать точкой с запятой (;). Если этого не сделать, то клиент будет висеть и ждать продолжения. `sqlalchemy` не требует точки с запятой и сам говорит клиенту когда запрос закончился*

Так же забегая вперёд, для того, что бы избавиться от проблемы с дублями применяется такая логика настроенная под ситуацию (то есть банально проводим сортировку по дублям и выбираем первый):
```sql
SELECT * FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY email
               ORDER BY updated_at DESC, id DESC
           ) AS rn
    FROM users
) t
WHERE rn = 1;
```

In [2]:
from sqlalchemy import create_engine, text
import pandas as pd

In [3]:
engine = create_engine("sqlite:///my.db")          # База созданная вручную, для демонстрации всего остального
engine2 = create_engine("sqlite:///players.db")    # Готовая большая база, для демонстрации SELECT запросов

## Ключи

### PRIMARY KEY — первичный ключ

**Первичный ключ (PRIMARY KEY)** — это столбец таблицы, значение которого однозначно идентифицирует каждую строку и при этом:
- уникально
- не может быть NULL
- в таблице он один

### FOREIGN KEY — внешний ключ

**Внешний ключ (FOREIGN KEY)** — это ссылка на первичный ключ другой таблицы. Нужен, чтобы связать таблицы

Пример:  
**Таблица «Клиенты»**
| id (PRIMARY KEY) | name  |
|------------------|-------|
| 1                | Анна  |
| 2                | Борис |
| 3                | Вера  |

**Таблица «Заказы»**
| id  | client_id (FOREIGN KEY) | dish  |
|-----|-------------------------|-------|
| 10  | 1                       | Пицца |
| 11  | 1                       | Суп   |
| 12  | 3                       | Стейк |

- Заказ 10 → клиент 1 (Анна)
- Заказ 11 → клиент 1 (Анна)
- Заказ 12 → клиент 3 (Вера)

## Типы данных

**Числа:**
- INTEGER(или INT) — целые числа (42)
- DECIMAL(p, s) — точные числа с запятой (для денег) (DECIMAL(5, 2) = 199.99)
    - p - всего значащих цифр в числе, считая и до, и после запятой
    - s - ровно цифр после запятой
- FLOAT / REAL — приближённые дробные (3.14159)

**Строки:**
- VARCHAR(n) — строка до n символов (VARCHAR(4) = 'Анна')
- TEXT — длинный текст ('Однажды в студёную зимнюю пору я из лесу вышел...')

**Дата и время:**
- DATE — дата ('2026-10-16')
- TIMESTAMP — дата + время ('2026-10-16 14:30:00')

**Логический:**
- BOOLEAN — TRUE / FALSE

**Бинарные:**
- BLOB — байты, файлы, картинки (\x89PNG\r\n\x1a\n...) (\xFF\xD8\xFF\xE0...)

Так же есть `NULL` - это не тип данных, а одно из возможных значений, которое обозначает отсутствие данных

## Ограничения

- PRIMARY KEY - создаёт первичный ключ (id INTEGER PRIMARY KEY)
- NOT NULL - выдаёт ошибку, если оставить столбец пустым
- UNIQUE - выдаёт ошибку, если значение в столбце не уникально
- DEFAULT - значение по умолчанию (is_active BOOLEAN DEFAULT TRUE)
- REFERENCES - создаёт вторичный ключ (user_id INTEGER REFERENCES users(id), где users - другая таблица в той же bd, id - название столбца с первичным ключём в таблице users)

## CREATE TABLE — создать таблицу

- `CREATE TABLE users` - создаёт новую таблицу с именем users
    - шаблон создания любого столбца: `имя_столбца  ТИП  ОГРАНИЧЕНИЯ`

In [4]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            age INTEGER,
            city TEXT)
        """))

*Есть конструкция "CREATE TABLE adults AS SELECT ..." которая позволяет из SELECT создать новую таблицу*

## INSERT — вставить данные

`users` - таблица куда вставляем, а кортеж названий столбцов и кортеж значений должны совпадать. Можно какие-то столбцы опускать и не записывать, тогда в них будет NULL (если только не стоит ограничение NOT NULL):

In [6]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO users (name, age, city) VALUES ('Аня', 25, 'Москва');
         """))

In [7]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO users (name, age, city) VALUES
        ('Борис', 30, 'Питер'),
        ('Вика', 28, 'Москва');
        """))

## SELECT — читать данные

`SELECT` - самая сложная по структуре команда в sql, поэтому стоит внимательно следить за последовательностью её команд, там есть строгая иерархия:
| № | Часть       | Что делает                          | Обязательна? |
|---|-------------|-------------------------------------|--------------|
| 1 | `SELECT`    | Что взять (столбцы)                 | Да           |
| 2 | `FROM`      | Откуда взять (таблица)              | Да           |
| 3 | `JOIN`      | С чем соединить                     | Нет          |
| 4 | `WHERE`     | Какие строки оставить (фильтр)      | Нет          |
| 5 | `GROUP BY`  | Как сгруппировать                   | Нет          |
| 6 | `HAVING`    | Какие группы оставить (фильтр)      | Нет          |
| 7 | `ORDER BY`  | Как отсортировать                   | Нет          |
| 8 | `LIMIT`     | Сколько строк вывести               | Нет          |

НО! Что важно, не смотря на такой порядок написания, выполнение идёт в совершенно другом порядке. Это важно учитывать при написании запроса!:
| № | Часть       | Что делает                          |
|---|-------------|-------------------------------------|
| 1 | `FROM`      | Берёт таблицу(ы)                    |
| 2 | `JOIN`      | Соединяет таблицы                   |
| 3 | `WHERE`     | Фильтрует строки                    |
| 4 | `GROUP BY`  | Группирует строки                   |
| 5 | `HAVING`    | Фильтрует группы                    |
| 6 | `SELECT`    | Выбирает столбцы, считает оконки    |
| 7 | `ORDER BY`  | Сортирует результат                 |
| 8 | `LIMIT`     | Ограничивает количество строк       |

**Читаем вообще всё (самый простой вариант):**

In [4]:
df = pd.read_sql(text("SELECT * FROM players"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,1,Al,Pratt,NaN,R,140.0,67.0,1871-05-04,1872-08-19,1847.0,11.0,19.0,Pittsburgh,PA,USA
1,2,Art,Allison,NaN,NaN,150.0,68.0,1871-05-04,1876-10-05,1849.0,1.0,29.0,Philadelphia,PA,USA
2,3,Bill,Kelly,NaN,NaN,150.0,67.0,1871-05-04,1871-08-29,NaN,NaN,NaN,New York,NY,USA
3,4,Bill,Lennon,NaN,NaN,145.0,67.0,1871-05-04,1873-07-11,1845.0,1.0,3.0,Brooklyn,NY,USA
4,5,Bobby,Mathews,R,R,140.0,65.0,1871-05-04,1887-10-10,1851.0,11.0,21.0,Baltimore,MD,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20764,20765,William,Hulbert,NaN,NaN,NaN,NaN,NaN,NaN,1832.0,10.0,23.0,Burlington Flats,NY,USA
20765,20766,William,Hyndman,NaN,NaN,150.0,68.0,NaN,NaN,1854.0,10.0,1.0,Philadelphia,PA,USA
20766,20767,William,Stein,NaN,L,NaN,NaN,NaN,NaN,1868.0,4.0,9.0,Telford,PA,USA
20767,20768,William,Sullivan,NaN,NaN,NaN,NaN,NaN,NaN,1864.0,3.0,12.0,St. Louis,MO,USA


*Очень часто используют $*$ что бы указать выбрать всё*

**Читаем конкретные столбцы:**

In [6]:
df = pd.read_sql(text("SELECT first_name, last_name FROM players"), engine2)
df

,first_name,last_name
0,Al,Pratt
1,Art,Allison
2,Bill,Kelly
3,Bill,Lennon
4,Bobby,Mathews
...,...,...
20764,William,Hulbert
20765,William,Hyndman
20766,William,Stein
20767,William,Sullivan


## UPDATE — изменить данные

`users` - таблица куда вставляем. Далее указываем "столбец = новое значение". Причём если не добавить `WHERE`, то изменится вообще весь столбец:

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE users SET city = 'Сочи',
                         age = 31,
                         name = 'Анна' 
                WHERE name = 'Аня'
        """))

## DELETE — удалить данные

Просто добавление строки, для примера её удаления:

In [56]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO users (name, age, city) VALUES ('Рыжий', 250, 'Юпитер');
         """))

df = pd.read_sql(text("SELECT * FROM users"), engine)
df

,id,name,age,city
0,1,Анна,31,Сочи
1,2,Борис,30,Питер
2,3,Вика,28,Москва
3,4,Рыжий,250,Юпитер


Тут такая же ситуация, без `WHERE` удалится вообще всё:

In [57]:
with engine.begin() as conn:
    conn.execute(text("DELETE FROM users WHERE name = 'Рыжий'"))

df = pd.read_sql(text("SELECT * FROM users"), engine)
df

,id,name,age,city
0,1,Анна,31,Сочи
1,2,Борис,30,Питер
2,3,Вика,28,Москва


## WHERE — фильтрация

Применяется в основном в SELECT, UPDATE, DELETE. Фильтрует столбцы по определённым значениям. Фильтраций может быть множество и делятся они засчёт следующих стандартных логических операторов:
| Оператор      | Значение       |
|---------------|----------------|
| `=`           | равно          |
| `<>` или `!=` | не равно       |
| `>` `<` `>=` `<=` | сравнения  |
| `AND`         | И              |
| `OR`          | ИЛИ            |
| `NOT`         | НЕ             |

In [8]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight >= 190 AND birth_city = 'New York'"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,123,Bill,Boyd,NaN,NaN,250,NaN,1872-04-22,1875-10-09,1852,12.0,22.0,New York,NY,USA
1,157,Jim,Holdsworth,R,R,195,72.0,1872-05-14,1884-06-09,1850,7.0,14.0,New York,NY,USA
2,514,John,Smith,NaN,NaN,210,71.0,1882-05-01,1882-09-27,1858,NaN,NaN,New York,NY,USA
3,632,Dave,Orr,R,R,250,71.0,1883-05-17,1890-10-02,1859,9.0,29.0,New York,NY,USA
4,634,Ren,Deagle,R,R,190,69.0,1883-05-17,1884-09-12,1858,6.0,26.0,New York,NY,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,17728,Dellin,Betances,R,R,265,80.0,2011-09-22,2021-04-07,1988,3.0,23.0,New York,NY,USA
61,18783,Jeimer,Candelario,B,R,222,74.0,2016-07-03,2023-05-31,1993,11.0,24.0,New York,NY,USA
62,19318,James,Norwood,R,R,215,74.0,2018-07-11,2022-06-12,1993,12.0,24.0,New York,NY,USA
63,19782,Luis,Garcia,L,R,220,74.0,2020-08-14,2023-05-31,2000,5.0,16.0,New York,NY,USA


## IN, BETWEEN, LIKE, IS

### IN

Выбор из списка:

In [9]:
df = pd.read_sql(text("SELECT * FROM players WHERE birth_city IN ('New York', 'Brooklyn')"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,3,Bill,Kelly,NaN,NaN,150.0,67.0,1871-05-04,1871-08-29,NaN,NaN,NaN,New York,NY,USA
1,4,Bill,Lennon,NaN,NaN,145.0,67.0,1871-05-04,1873-07-11,1845.0,1.0,3.0,Brooklyn,NY,USA
2,6,Charlie,Pabor,L,L,155.0,68.0,1871-05-04,1875-10-28,1846.0,9.0,24.0,Brooklyn,NY,USA
3,13,James,McDermott,NaN,NaN,NaN,NaN,1871-05-04,1872-07-06,1846.0,NaN,NaN,Brooklyn,NY,USA
4,17,Tom,Carey,R,R,145.0,68.0,1871-05-04,1879-09-29,1846.0,11.0,2.0,Brooklyn,NY,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,20662,Jacob,Ruppert,NaN,NaN,NaN,NaN,NaN,NaN,1867.0,8.0,5.0,New York,NY,USA
515,20668,Jim,Kennedy,NaN,NaN,NaN,NaN,NaN,NaN,1862.0,4.0,NaN,New York,NY,USA
516,20671,Jim,Price,NaN,NaN,NaN,NaN,NaN,NaN,1847.0,9.0,20.0,New York,NY,USA
517,20713,Matt,Galante,R,R,157.0,66.0,NaN,NaN,1944.0,3.0,22.0,Brooklyn,NY,USA


### BETWEEN

Выбор из диапазона(включительно):

In [14]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight BETWEEN 200 AND 205"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,270,Charley,Jones,R,R,202,71,1875-05-04,1888-04-26,1852,4.0,30.0,Alamance County,NC,USA
1,397,Bill,Phillips,R,R,202,72,1879-05-01,1888-10-14,1857,4.0,NaN,St. John,NB,CAN
2,496,Billy,Taylor,R,R,204,71,1881-05-21,1887-08-16,1855,NaN,NaN,Washington,DC,USA
3,650,Edgar,Smith,NaN,R,200,72,1883-06-20,1883-09-29,1860,10.0,15.0,North Haven,CT,USA
4,688,Ed,Crane,R,R,204,70,1884-04-17,1893-07-19,1862,5.0,27.0,Boston,MA,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2088,20547,Ryan,Walker,R,R,200,74,2023-05-21,2023-05-31,1995,11.0,26.0,Arlington,WA,USA
2089,20577,Billy,Evans,NaN,NaN,205,71,NaN,NaN,1884,2.0,10.0,Chicago,IL,USA
2090,20708,Louis,Santop,L,R,200,74,NaN,NaN,1889,1.0,17.0,Fort Worth,TX,USA
2091,20738,Ray,Miller,R,R,200,75,NaN,NaN,1945,4.0,30.0,Takoma Park,MD,USA


### LIKE

Поиск по шаблону:
| Символ | Значение |
|--------|----------|
| `%`    | любое количество символов (в том числе ноль) |
| `_`    | ровно один любой символ |

*Причём важно понимать, комбинация вроде "А_" - обозначает ровно 2 символа*

In [15]:
df = pd.read_sql(text("SELECT * FROM players WHERE first_name LIKE 'Bil%'"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,3,Bill,Kelly,NaN,NaN,150.0,67.0,1871-05-04,1871-08-29,NaN,NaN,NaN,New York,NY,USA
1,4,Bill,Lennon,NaN,NaN,145.0,67.0,1871-05-04,1873-07-11,1845.0,1.0,3.0,Brooklyn,NY,USA
2,56,Bill,Craver,R,R,160.0,69.0,1871-05-09,1877-10-05,1844.0,6.0,NaN,Troy,NY,USA
3,98,Bill,Stearns,NaN,R,NaN,NaN,1871-06-26,1875-06-12,1853.0,3.0,20.0,Washington,DC,USA
4,102,Bill,Barrett,NaN,NaN,NaN,NaN,1871-07-08,1873-10-18,NaN,NaN,NaN,Baltimore,MD,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,20574,Bill,Sharsig,NaN,NaN,NaN,NaN,NaN,NaN,1855.0,1.0,1.0,Philadelphia,PA,USA
668,20575,Bill,Shettsline,NaN,NaN,NaN,NaN,NaN,NaN,1863.0,10.0,25.0,Philadelphia,PA,USA
669,20576,Bill,Veeck,NaN,NaN,NaN,NaN,NaN,NaN,1914.0,2.0,9.0,Chicago,IL,USA
670,20577,Billy,Evans,NaN,NaN,205.0,71.0,NaN,NaN,1884.0,2.0,10.0,Chicago,IL,USA


### IS

В основном применяется исключительно для `NULL`, но в редких случая можно писать допустим к булевому случаю `WHERE name IS TRUE`. В данном примере запрос вывел бы все строки, где нету значения веса:

In [16]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight IS NULL"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,9,Elmer,White,NaN,NaN,None,NaN,1871-05-04,1871-09-27,1849.0,12.0,7.0,Caton,NY,USA
1,13,James,McDermott,NaN,NaN,None,NaN,1871-05-04,1872-07-06,1846.0,NaN,NaN,Brooklyn,NY,USA
2,28,Frank,Norton,NaN,NaN,None,NaN,1871-05-05,1871-05-05,1845.0,6.0,9.0,Port Jefferson,NY,USA
3,67,Pete,Donnelly,NaN,NaN,None,NaN,1871-05-13,1871-07-08,NaN,NaN,NaN,NaN,NaN,NaN
4,83,Frank,Barrows,NaN,NaN,None,NaN,1871-05-20,1871-10-07,1844.0,10.0,22.0,Hudson,OH,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
804,20763,Watch,Burnham,NaN,NaN,None,NaN,NaN,NaN,1860.0,5.0,20.0,Albion,MI,USA
805,20764,Will,Harridge,NaN,NaN,None,NaN,NaN,NaN,1881.0,10.0,16.0,Chicago,IL,USA
806,20765,William,Hulbert,NaN,NaN,None,NaN,NaN,NaN,1832.0,10.0,23.0,Burlington Flats,NY,USA
807,20767,William,Stein,NaN,L,None,NaN,NaN,NaN,1868.0,4.0,9.0,Telford,PA,USA


Конечно куда полезнее использовать следующим образом:

In [19]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight IS NOT NULL"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,1,Al,Pratt,NaN,R,140,67.0,1871-05-04,1872-08-19,1847.0,11.0,19.0,Pittsburgh,PA,USA
1,2,Art,Allison,NaN,NaN,150,68.0,1871-05-04,1876-10-05,1849.0,1.0,29.0,Philadelphia,PA,USA
2,3,Bill,Kelly,NaN,NaN,150,67.0,1871-05-04,1871-08-29,NaN,NaN,NaN,New York,NY,USA
3,4,Bill,Lennon,NaN,NaN,145,67.0,1871-05-04,1873-07-11,1845.0,1.0,3.0,Brooklyn,NY,USA
4,5,Bobby,Mathews,R,R,140,65.0,1871-05-04,1887-10-10,1851.0,11.0,21.0,Baltimore,MD,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19955,20756,Turkey,Stearnes,L,L,175,71.0,NaN,NaN,1901.0,5.0,8.0,Nashville,TN,USA
19956,20757,Vedie,Himsl,R,R,180,73.0,NaN,NaN,1917.0,4.0,2.0,Plevna,MT,USA
19957,20758,Vern,Rapp,R,R,195,72.0,NaN,NaN,1928.0,5.0,11.0,St. Louis,MO,USA
19958,20766,William,Hyndman,NaN,NaN,150,68.0,NaN,NaN,1854.0,10.0,1.0,Philadelphia,PA,USA


## ORDER BY — сортировка

По возрастанию:

In [18]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight IS NOT NULL ORDER BY weight"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,8418,Eddie,Gaedel,R,L,65,43.0,1951-08-19,1951-08-19,1925.0,6.0,8.0,Chicago,IL,USA
1,124,Candy,Cummings,R,R,120,69.0,1872-04-22,1877-08-18,1848.0,10.0,18.0,Ware,MA,USA
2,1358,Sparrow,McCaffrey,NaN,NaN,120,NaN,1889-08-13,1889-08-15,1868.0,NaN,NaN,Philadelphia,PA,USA
3,2277,George,Leitner,L,R,120,67.0,1901-06-29,1902-08-25,1871.0,6.0,19.0,Parkton,MD,USA
4,1052,Bill,McCaffrey,NaN,NaN,121,NaN,1885-06-15,1885-06-15,1862.0,4.0,1.0,Baltimore County,MD,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19955,5570,Jumbo,Brown,R,R,295,76.0,1925-08-26,1941-08-27,1907.0,4.0,30.0,Greene,RI,USA
19956,14567,Dmitri,Young,B,R,295,74.0,1996-08-29,2008-07-11,1973.0,10.0,11.0,Vicksburg,MS,USA
19957,15440,CC,Sabathia,L,L,300,78.0,2001-04-08,2019-09-24,1980.0,7.0,21.0,Vallejo,CA,USA
19958,18277,Jumbo,Diaz,R,R,315,76.0,2014-06-20,2017-07-16,1984.0,2.0,27.0,La Romana,La Romana,D.R.


По убыванию:

In [20]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight IS NOT NULL ORDER BY weight DESC"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,16384,Walter,Young,L,R,320,77.0,2005-09-06,2005-10-02,1980.0,2.0,18.0,Hattiesburg,MS,USA
1,18277,Jumbo,Diaz,R,R,315,76.0,2014-06-20,2017-07-16,1984.0,2.0,27.0,La Romana,La Romana,D.R.
2,15440,CC,Sabathia,L,L,300,78.0,2001-04-08,2019-09-24,1980.0,7.0,21.0,Vallejo,CA,USA
3,5570,Jumbo,Brown,R,R,295,76.0,1925-08-26,1941-08-27,1907.0,4.0,30.0,Greene,RI,USA
4,14567,Dmitri,Young,B,R,295,74.0,1996-08-29,2008-07-11,1973.0,10.0,11.0,Vicksburg,MS,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19955,1052,Bill,McCaffrey,NaN,NaN,121,NaN,1885-06-15,1885-06-15,1862.0,4.0,1.0,Baltimore County,MD,USA
19956,124,Candy,Cummings,R,R,120,69.0,1872-04-22,1877-08-18,1848.0,10.0,18.0,Ware,MA,USA
19957,1358,Sparrow,McCaffrey,NaN,NaN,120,NaN,1889-08-13,1889-08-15,1868.0,NaN,NaN,Philadelphia,PA,USA
19958,2277,George,Leitner,L,R,120,67.0,1901-06-29,1902-08-25,1871.0,6.0,19.0,Parkton,MD,USA


Двойная сортировка (сначала по 1, потом по 2):

In [25]:
df = pd.read_sql(text("SELECT * FROM players WHERE weight IS NOT NULL AND height IS NOT NULL ORDER BY weight DESC, height DESC"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,16384,Walter,Young,L,R,320,77,2005-09-06,2005-10-02,1980.0,2.0,18.0,Hattiesburg,MS,USA
1,18277,Jumbo,Diaz,R,R,315,76,2014-06-20,2017-07-16,1984.0,2.0,27.0,La Romana,La Romana,D.R.
2,15440,CC,Sabathia,L,L,300,78,2001-04-08,2019-09-24,1980.0,7.0,21.0,Vallejo,CA,USA
3,5570,Jumbo,Brown,R,R,295,76,1925-08-26,1941-08-27,1907.0,4.0,30.0,Greene,RI,USA
4,14567,Dmitri,Young,B,R,295,74,1996-08-29,2008-07-11,1973.0,10.0,11.0,Vicksburg,MS,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19901,6643,Pat,Ankenman,R,R,125,64,1936-04-16,1944-06-02,1912.0,12.0,23.0,Houston,TX,USA
19902,1764,Yale,Murphy,L,R,125,63,1894-04-19,1897-07-26,1869.0,10.0,11.0,Southborough,MA,USA
19903,124,Candy,Cummings,R,R,120,69,1872-04-22,1877-08-18,1848.0,10.0,18.0,Ware,MA,USA
19904,2277,George,Leitner,L,R,120,67,1901-06-29,1902-08-25,1871.0,6.0,19.0,Parkton,MD,USA


## LIMIT — ограничить количество строк

In [27]:
df = pd.read_sql(text("SELECT * FROM players LIMIT 3"), engine2)
df

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,1,Al,Pratt,None,R,140,67,1871-05-04,1872-08-19,1847.0,11.0,19.0,Pittsburgh,PA,USA
1,2,Art,Allison,None,NaN,150,68,1871-05-04,1876-10-05,1849.0,1.0,29.0,Philadelphia,PA,USA
2,3,Bill,Kelly,None,NaN,150,67,1871-05-04,1871-08-29,NaN,NaN,NaN,New York,NY,USA


## Агрегатные функции

**Агрегатные функции** - считают что-то по множеству строк

| Функция | Что делает |
|---------|-----------|
| `COUNT(*)` | Считает количество строк |
| `COUNT(столбец)` | Считает непустые значения в столбце (NULL не считает) |
| `SUM(столбец)` | Сумма значений |
| `AVG(столбец)` | Среднее значение |
| `MIN(столбец)` | Минимальное значение |
| `MAX(столбец)` | Максимальное значение |

In [28]:
df = pd.read_sql(text("SELECT AVG(weight) FROM players"), engine2)
df

,AVG(weight)
0,188.346092


## GROUP BY — группировка

В SELECT при GROUP BY можно выводить только два типа столбцов:
- Те, что стоят в GROUP BY
- Агрегатные функции (COUNT, SUM, AVG, MIN, MAX)

То есть если брать тот же столбец по которому группировали, то покажет лишь уникальные значения:

In [29]:
df = pd.read_sql(text("SELECT birth_city FROM players GROUP BY birth_city"), engine2)
df

,birth_city
0,NaN
1,Aalen
2,Aberdeen
3,Abilene
4,Abingdon
...,...
4983,Zeigler
4984,Zeist
4985,Zephyrhills
4986,Zerbe


В противном случаи, нужно указать что делать с конкретным столбцом:

In [30]:
df = pd.read_sql(text("SELECT birth_city, AVG(weight) FROM players GROUP BY birth_city"), engine2)
df

,birth_city,AVG(weight)
0,NaN,171.115942
1,Aalen,180.000000
2,Aberdeen,182.111111
3,Abilene,197.555556
4,Abingdon,195.000000
...,...,...
4983,Zeigler,172.000000
4984,Zeist,200.000000
4985,Zephyrhills,225.000000
4986,Zerbe,151.000000


## HAVING — фильтр для групп

Логика та же самая, что и у `WHERE`, только применяет он исключительно для данных после группировки:

In [46]:
df = pd.read_sql(text("SELECT birth_city, AVG(weight) AS avg FROM players GROUP BY birth_city HAVING avg > 200"), engine2)
df

,birth_city,avg
0,Acarigua,226.000000
1,Ada,200.833333
2,Adel,225.000000
3,Advance,235.000000
4,Agana Heights,235.000000
...,...,...
925,Yorba Linda,202.000000
926,Yreka,205.000000
927,Yucaipa,238.333333
928,Zeeland,202.500000


## JOIN — соединение таблиц по общему полю

| Тип JOIN | Что возвращает |
|----------|----------------|
| `INNER JOIN` | Только строки с совпадением в обеих таблицах |
| `LEFT JOIN` | Все строки из левой + совпадения из правой (иначе NULL) |
| `RIGHT JOIN` | Все строки из правой + совпадения из левой (иначе NULL) |
| `FULL JOIN` | Все строки из обеих таблиц (иначе NULL) |
| `CROSS JOIN` | Все возможные комбинации строк (декартово произведение) |

In [38]:
df = pd.read_sql(text("SELECT u.name, u.age, e.Evaluation FROM users u INNER JOIN Evaluations e ON u.name = e.name"), engine)
df

,name,age,Evaluation
0,Анна,31,4
1,Анна,31,5
2,Борис,30,3


In [39]:
df = pd.read_sql(text("SELECT u.name, u.age, e.Evaluation FROM users u LEFT JOIN Evaluations e ON u.name = e.name"), engine)
df

,name,age,Evaluation
0,Анна,31,4.0
1,Анна,31,5.0
2,Борис,30,3.0
3,Вика,28,NaN


In [40]:
df = pd.read_sql(text("SELECT u.name, u.age, e.Evaluation FROM users u RIGHT JOIN Evaluations e ON u.name = e.name"), engine)
df

,name,age,Evaluation
0,Анна,31.0,4
1,Анна,31.0,5
2,Борис,30.0,3
3,NaN,NaN,5


In [45]:
df = pd.read_sql(text("SELECT u.name, u.age, e.Evaluation FROM users AS u FULL JOIN Evaluations AS e ON u.name = e.name"), engine)
df

,name,age,Evaluation
0,Анна,31.0,4.0
1,Анна,31.0,5.0
2,Борис,30.0,3.0
3,Вика,28.0,NaN
4,NaN,NaN,5.0


## Псевдонимы (AS)

Очень часто используются когда либо нужно сделать названия столбцов более понятными, либо дать название агрегатной функции. Так же часто используют при группировки что бы общие названия не конфликтовали:

In [ ]:
df = pd.read_sql(text("SELECT name as имя, age AS возраст FROM users"), engine)
df

,имя,возраст
0,Анна,31
1,Борис,30
2,Вика,28


Псевдонимы можно давать результату операции:

In [ ]:
df = pd.read_sql(text("SELECT age, age * age AS age_2 FROM users"), engine)
df

,age,age_2
0,31,961
1,30,900
2,28,784


## DISTINCT — уникальные значения

In [58]:
df = pd.read_sql(text("SELECT DISTINCT birth_city FROM players"), engine2)
df

,birth_city
0,Pittsburgh
1,Philadelphia
2,New York
3,Brooklyn
4,Baltimore
...,...
4983,McDonald Township
4984,Plevna
4985,Windham
4986,Tiskilwa


## Подзапросы и CTE (WITH) <a id="cte-section"></a>

### Подзапрос

**Подзапрос** — это `SELECT` внутри другого запроса. Он выполняется первым, а его результат используется внешним запросом

**Подзапрос в WHERE**. Тут мы выводим таблицу с игроками у которых вес больше, чем среднее значение веса всех игроков:

In [59]:
df = pd.read_sql(text("SELECT first_name, weight FROM players WHERE weight > (SELECT AVG(weight) FROM players)"), engine2)
df

,first_name,weight
0,Cap,227
1,Bill,250
2,John,194
3,Jack,189
4,Jim,195
...,...,...
9124,Rube,230
9125,Terry,190
9126,Tony,190
9127,Trent,195


**Подзапрос в FROM — «таблица внутри таблицы»**. Например, мы сначала (внутри FROM) сгруппировали по городам и к каждому городу нашли средний вес. А затем исходя из получившейся таблицы, нашли те средние значения, которые оказались больше 200:

In [65]:
df = pd.read_sql(text("""SELECT birth_city, avg 
                        FROM(SELECT birth_city, AVG(weight) AS avg FROM players GROUP BY birth_city) 
                        WHERE avg > 200"""), engine2)
df

,birth_city,avg
0,Acarigua,226.000000
1,Ada,200.833333
2,Adel,225.000000
3,Advance,235.000000
4,Agana Heights,235.000000
...,...,...
925,Yorba Linda,202.000000
926,Yreka,205.000000
927,Yucaipa,238.333333
928,Zeeland,202.500000


### CTE (WITH)

**CTE (Common Table Expression)** — это именованный подзапрос, который пишется в начале запроса через WITH. По сути — тот же подзапрос в FROM, но:
- Имеет имя — можно ссылаться несколько раз
- Пишется сверху — читается сверху вниз, как код
- Может быть несколько — через запятую

Пример того же самого подзапроса, что был выше, но через CTE:

In [ ]:
df = pd.read_sql(text("""WITH city_stats AS (SELECT birth_city, AVG(weight) AS avg FROM players GROUP BY birth_city) 
                        SELECT birth_city, avg 
                        FROM(city_stats) 
                        WHERE avg > 200"""), engine2)
df

,birth_city,avg
0,Acarigua,226.000000
1,Ada,200.833333
2,Adel,225.000000
3,Advance,235.000000
4,Agana Heights,235.000000
...,...,...
925,Yorba Linda,202.000000
926,Yreka,205.000000
927,Yucaipa,238.333333
928,Zeeland,202.500000


**Несколько CTE в одном запросе + актуально с использованием JOIN**. То есть первые два WITH повторяют запрос выше, но затем в конце мы выводим только те города, где средняя по росту была больше 200 через JOIN нашей кастомной таблицы и оригинальной:

In [66]:
df = pd.read_sql(text("""WITH 
                        city_stats AS(
                            SELECT birth_city, AVG(weight) AS avg
                            FROM players
                            GROUP BY birth_city),
                        big_cities AS(
                            SELECT birth_city
                            FROM city_stats
                            WHERE avg > 200)
                        SELECT p.first_name, p.birth_city
                        FROM players AS p
                        JOIN big_cities AS b ON p.birth_city = b.birth_city
                            """), engine2)
df

,first_name,birth_city
0,Cap,Marshalltown
1,William,Parma
2,John,Ithaca
3,Ham,Augusta
4,Harry,Pocahontas
...,...,...
3198,Luis,Santo Domingo
3199,Nick,Ontario
3200,Stump,Brunswick
3201,Trey,Amarillo


## Оконные функции (Window Functions) (OVER) <a id="over"></a>

**Оконные функции** - считают что-то по группе, но оставляют каждую строку на месте. То есть как видно в примере ниже, для каждого города подсчитана своя статистика, при этом количество строк в таблице вообще не поменялось

**Синтаксис оконной функции:**
| Часть | Что делает |
|-------|------------|
| `PARTITION BY столбец, ...` | Разбивает строки на группы (разделы). Без неё окно = вся таблица |
| `ORDER BY столбец [ASC\|DESC], ...` | Задаёт порядок внутри раздела. Нужен для ранжирования (`ROW_NUMBER`, `RANK`, `LAG`, `LEAD`); для агрегатов — чтобы получить накопительный итог |
| `ROWS \| RANGE \| GROUPS BETWEEN ... AND ...` | Задаёт рамку окна — какие именно строки видит функция для текущей строки. Без неё рамка = весь раздел (если нет `ORDER BY`) или «от начала раздела до текущей строки» (если `ORDER BY` есть) |

*Когда говорят о* **скользящем окне** *то имеют ввиду именно ROWS, ибо он проходится по каждому текущему значению не меняя свой изначальный размер. То есть допустим может быть ровно 3 [предыдущий, текущий, следующий]*

**Категории оконных функций:**

### Агрегатные

Используются классические агрегатные функции, которые обычно используют при группировки (SUM, AVG, COUNT, MIN, MAX):

In [70]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                SUM(weight) OVER (PARTITION BY birth_city) AS total_weight,
                                COUNT(*) OVER (PARTITION BY birth_city) AS cnt,
                                AVG(weight) OVER (PARTITION BY birth_city) AS avg_weight
                        FROM players
                        WHERE birth_city IS NOT NULL
                            """), engine2)
df

,first_name,birth_city,weight,total_weight,cnt,avg_weight
0,Otto,Aalen,180.0,180.0,1,180.000000
1,Frank,Aberdeen,NaN,1639.0,10,182.111111
2,George,Aberdeen,189.0,1639.0,10,182.111111
3,Tony,Aberdeen,180.0,1639.0,10,182.111111
4,Guy,Aberdeen,175.0,1639.0,10,182.111111
...,...,...,...,...,...,...
20597,Domonic,Zephyrhills,225.0,225.0,1,225.000000
20598,Sparky,Zerbe,151.0,151.0,1,151.000000
20599,Red,Zion,176.0,536.0,3,178.666667
20600,Danny,Zion,160.0,536.0,3,178.666667


**ВАЖНО!** С `ORDER BY` агрегатные функции ведут себя не типично! То есть к примеру, вместо того, что бы посчитать сумму во всём окне, агрегатнная функция будет проходиться по порядку и выдавать **накопленную сумму**:

In [6]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                SUM(weight) OVER (PARTITION BY birth_city ORDER BY weight) AS total_weight
                        FROM players
                        WHERE birth_city IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,total_weight
0,Otto,Aalen,180.0,180.0
1,Frank,Aberdeen,NaN,NaN
2,Justin,Aberdeen,165.0,165.0
3,Guy,Aberdeen,175.0,515.0
4,Cal,Aberdeen,175.0,515.0
5,Tony,Aberdeen,180.0,695.0
6,Pete,Aberdeen,185.0,880.0
7,George,Aberdeen,189.0,1069.0
8,John,Aberdeen,190.0,1639.0
9,Terry,Aberdeen,190.0,1639.0


*Можно заметить, что тут немного странно себя ведёт накопление. Это связанно с тем, что по умолчанию используется `RANGE`. Поэтому для стандартного накопления нужно дописать `ROWS`. Подробнее об этом ниже*

### Ранжирующие

*По сути все три функции определяют как относится к повторам. 1. `ROW_NUMBER()` - игнорировать и продолжать раздавать значение. 2. `RANK()` - если значения совпали, дать один номер, но при этом продолжать с той цифры, как будто игнорировали. 3. `DENSE_RANK()` - если значения совпали, дать один номер и затем продолжать как обычно. 4. `NTILE(n)` - разделить на ровно указанное количество частей (можно меньше, но не больше)*

**ROW_NUMBER()** — просто номер строки: 1, 2, 3, 4 (без повторов):

In [76]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                ROW_NUMBER() OVER (PARTITION BY birth_city ORDER BY weight DESC) AS rn
                        FROM players
                        WHERE birth_city IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,rn
0,Otto,Aalen,180.0,1
1,John,Aberdeen,190.0,1
2,Terry,Aberdeen,190.0,2
3,Karl,Aberdeen,190.0,3
4,George,Aberdeen,189.0,4
5,Pete,Aberdeen,185.0,5
6,Tony,Aberdeen,180.0,6
7,Guy,Aberdeen,175.0,7
8,Cal,Aberdeen,175.0,8
9,Justin,Aberdeen,165.0,9


*ПОЯСНЕНИЕ: суть тут в том, что мы как бы сначала после `PARTITION BY birth_city` получили несколько мини-таблиц. В каждой этой мини таблице мы применили `ORDER BY weight DESC` и уже затем в основной таблице записали номера. То есть тут получается у каждого города будет свой (1,2,3 ...)*

**RANK()** — как ROW_NUMBER, но при равенстве даёт одинаковый ранг и пропускает следующие: 1, 2, 2, 4:

In [75]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                RANK() OVER (PARTITION BY birth_city ORDER BY weight DESC) AS rn
                        FROM players
                        WHERE birth_city IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,rn
0,Otto,Aalen,180.0,1
1,John,Aberdeen,190.0,1
2,Terry,Aberdeen,190.0,1
3,Karl,Aberdeen,190.0,1
4,George,Aberdeen,189.0,4
5,Pete,Aberdeen,185.0,5
6,Tony,Aberdeen,180.0,6
7,Guy,Aberdeen,175.0,7
8,Cal,Aberdeen,175.0,7
9,Justin,Aberdeen,165.0,9


**DENSE_RANK()** — как RANK, но не пропускает: 1, 2, 2, 3:

In [77]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                DENSE_RANK() OVER (PARTITION BY birth_city ORDER BY weight DESC) AS rn
                        FROM players
                        WHERE birth_city IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,rn
0,Otto,Aalen,180.0,1
1,John,Aberdeen,190.0,1
2,Terry,Aberdeen,190.0,1
3,Karl,Aberdeen,190.0,1
4,George,Aberdeen,189.0,2
5,Pete,Aberdeen,185.0,3
6,Tony,Aberdeen,180.0,4
7,Guy,Aberdeen,175.0,5
8,Cal,Aberdeen,175.0,5
9,Justin,Aberdeen,165.0,6


**NTILE(n)** — делит на n примерно равных групп: 1, 2, 3, 4:

In [78]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                NTILE(3) OVER (PARTITION BY birth_city ORDER BY weight DESC) AS rn
                        FROM players
                        WHERE birth_city IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,rn
0,Otto,Aalen,180.0,1
1,John,Aberdeen,190.0,1
2,Terry,Aberdeen,190.0,1
3,Karl,Aberdeen,190.0,1
4,George,Aberdeen,189.0,1
5,Pete,Aberdeen,185.0,2
6,Tony,Aberdeen,180.0,2
7,Guy,Aberdeen,175.0,2
8,Cal,Aberdeen,175.0,3
9,Justin,Aberdeen,165.0,3


### Функции смещения (LAG / LEAD)

**Функции смещения** - смотрят на предыдущую или следующую строку в окне

**LAG(столбец, N)** — значение из строки на N позиций назад (N - по умолчанию 1):

In [87]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                LAG(weight) OVER (ORDER BY weight) AS prev
                        FROM players
                        WHERE weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,prev
0,Eddie,Chicago,65,NaN
1,Candy,Ware,120,65.0
2,Sparrow,Philadelphia,120,120.0
3,George,Parkton,120,120.0
4,Bill,Baltimore County,121,120.0
5,Yale,Southborough,125,121.0
6,George,Cincinnati,125,125.0
7,Johnny,Troy,125,125.0
8,Pat,Houston,125,125.0
9,Dave,New York,126,125.0


**LEAD(столбец, N)** — значение из строки на N позиций вперёд (N - по умолчанию 1):

In [88]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                LEAD(weight) OVER (ORDER BY weight) AS prev
                        FROM players
                        WHERE weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,prev
0,Eddie,Chicago,65,120.0
1,Candy,Ware,120,120.0
2,Sparrow,Philadelphia,120,120.0
3,George,Parkton,120,121.0
4,Bill,Baltimore County,121,125.0
5,Yale,Southborough,125,125.0
6,George,Cincinnati,125,125.0
7,Johnny,Troy,125,125.0
8,Pat,Houston,125,126.0
9,Dave,New York,126,126.0


Более сложные пример использования, когда мы хотим отследить изменения во временных рядах (в нашем случаи таких нет, но смысл понятен):

In [90]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                LAG(weight) OVER (ORDER BY weight) AS prev,
                                weight - LAG(weight) OVER (ORDER BY weight) AS delta
                        FROM players
                        WHERE weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,prev,delta
0,Eddie,Chicago,65,NaN,NaN
1,Candy,Ware,120,65.0,55.0
2,Sparrow,Philadelphia,120,120.0,0.0
3,George,Parkton,120,120.0,0.0
4,Bill,Baltimore County,121,120.0,1.0
5,Yale,Southborough,125,121.0,4.0
6,George,Cincinnati,125,125.0,0.0
7,Johnny,Troy,125,125.0,0.0
8,Pat,Houston,125,125.0,0.0
9,Dave,New York,126,125.0,1.0


*Это основа для признаков «изменение к предыдущему периоду»*

### Функции распределения

**FIRST_VALUE(столбец)** — первое значение в окне:

In [96]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                FIRST_VALUE(weight) OVER (PARTITION BY birth_city) AS fv
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(11)

,first_name,birth_city,weight,fv
0,Otto,Aalen,180,180
1,George,Aberdeen,189,189
2,Tony,Aberdeen,180,189
3,Guy,Aberdeen,175,189
4,Pete,Aberdeen,185,189
5,John,Aberdeen,190,189
6,Terry,Aberdeen,190,189
7,Karl,Aberdeen,190,189
8,Justin,Aberdeen,165,189
9,Cal,Aberdeen,175,189


**LAST_VALUE(столбец)** — последнее значение в окне:

In [95]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                LAST_VALUE(weight) OVER (PARTITION BY birth_city) AS lv
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(11)

,first_name,birth_city,weight,lv
0,Otto,Aalen,180,180
1,George,Aberdeen,189,175
2,Tony,Aberdeen,180,175
3,Guy,Aberdeen,175,175
4,Pete,Aberdeen,185,175
5,John,Aberdeen,190,175
6,Terry,Aberdeen,190,175
7,Karl,Aberdeen,190,175
8,Justin,Aberdeen,165,175
9,Cal,Aberdeen,175,175


**NTH_VALUE(столбец, N)** — N-е значение:

In [98]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                NTH_VALUE(weight, 5) OVER (PARTITION BY birth_city) AS lv
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(11)

,first_name,birth_city,weight,lv
0,Otto,Aalen,180,NaN
1,George,Aberdeen,189,190.0
2,Tony,Aberdeen,180,190.0
3,Guy,Aberdeen,175,190.0
4,Pete,Aberdeen,185,190.0
5,John,Aberdeen,190,190.0
6,Terry,Aberdeen,190,190.0
7,Karl,Aberdeen,190,190.0
8,Justin,Aberdeen,165,190.0
9,Cal,Aberdeen,175,190.0


**Просто популярная задача**: для каждого города — топ-3 самых высоких игроков:

In [100]:
df = pd.read_sql(text("""WITH ranked AS (
                            SELECT first_name, birth_city, height,
                                    ROW_NUMBER() OVER (PARTITION BY birth_city ORDER BY height DESC) AS rn
                            FROM players
                            WHERE birth_city IS NOT NULL AND height IS NOT NULL
                        )
                        SELECT first_name, birth_city, height
                        FROM ranked
                        WHERE rn < 4"""), engine2)
df.head(11)

,first_name,birth_city,height
0,Otto,Aalen,68
1,Karl,Aberdeen,76
2,Justin,Aberdeen,75
3,George,Aberdeen,73
4,John,Abilene,78
5,Hy,Abilene,76
6,Phil,Abilene,74
7,Gail,Abingdon,72
8,Gene,Abington,75
9,Dorn,Abington,74


**Помимо `PARTITION BY` и `ORDER BY` мы можем так же сузить само окно при помощи `ROWS | RANGE | GROUPS BETWEEN ... AND ...`**

**Границы рамки:**
| Граница | Значение | Для ROWS | Для RANGE | Для GROUPS |
|---------|----------|----------|-----------|------------|
| `UNBOUNDED PRECEDING` | От самого начала раздела | Первая строка раздела | Минимальное значение `ORDER BY` | Первая группа раздела |
| `N PRECEDING` | На N назад | N строк назад | Значение `ORDER BY − N` | N групп назад |
| `CURRENT ROW` | Текущая позиция | Текущая строка | Текущее значение `ORDER BY` | Текущая группа |
| `N FOLLOWING` | На N вперёд | N строк вперёд | Значение `ORDER BY + N` | N групп вперёд |
| `UNBOUNDED FOLLOWING` | До самого конца раздела | Последняя строка раздела | Максимальное значение `ORDER BY` | Последняя группа раздела |

### ROWS — по физическим строкам

**ROWS** — Считает строки штуками. «N строк назад», «N строк вперёд» — независимо от значений

In [8]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                SUM(weight) OVER (PARTITION BY birth_city ORDER BY weight ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS total_weight
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,total_weight
0,Otto,Aalen,180,180
1,Justin,Aberdeen,165,340
2,Guy,Aberdeen,175,515
3,Cal,Aberdeen,175,530
4,Tony,Aberdeen,180,540
5,Pete,Aberdeen,185,554
6,George,Aberdeen,189,564
7,John,Aberdeen,190,569
8,Terry,Aberdeen,190,570
9,Karl,Aberdeen,190,380


= «предыдущая строка + текущая + следующая». Ровно 3 строки, что бы там ни было в значениях

| weight | рамка (1 PRECEDING .. 1 FOLLOWING) |
|--------|-------------------------------------|
| 165    | [165, 175]        ← нет предыдущей |
| 175    | [165, 175, 175]                     |
| 175    | [175, 175, 180]                     |
| 180    | [175, 180]        ← нет следующей  |

### RANGE — по значениям ORDER BY

**RANGE** - считает не строки, а диапазон значений. «Все строки, у которых значение ORDER BY попадает в диапазон от X до Y»

In [9]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                SUM(weight) OVER (PARTITION BY birth_city ORDER BY weight RANGE BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS total_weight
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,total_weight
0,Otto,Aalen,180,180
1,Justin,Aberdeen,165,165
2,Guy,Aberdeen,175,350
3,Cal,Aberdeen,175,350
4,Tony,Aberdeen,180,180
5,Pete,Aberdeen,185,185
6,George,Aberdeen,189,759
7,John,Aberdeen,190,759
8,Terry,Aberdeen,190,759
9,Karl,Aberdeen,190,759


Для каждой строки берётся диапазон `[weight − 1, weight + 1]` и забираются **все** строки раздела, чьё значение попало внутрь

| weight | Диапазон `[w−1, w+1]` | Какие строки попадают | Сумма |
|--------|------------------------|------------------------|-------|
| 165    | [164, 166]             | только 165             | 165   |
| 175    | [174, 176]             | 175, 175               | 350   |
| 175    | [174, 176]             | 175, 175               | 350   |
| 180    | [179, 181]             | только 180             | 180   |

### GROUPS — по группам ровесников

**GROUPS** - компромисс между ROWS и RANGE. Считает группы строк с одинаковым значением ORDER BY

In [10]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                SUM(weight) OVER (PARTITION BY birth_city ORDER BY weight GROUPS BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS total_weight
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,total_weight
0,Otto,Aalen,180,180
1,Justin,Aberdeen,165,515
2,Guy,Aberdeen,175,695
3,Cal,Aberdeen,175,695
4,Tony,Aberdeen,180,715
5,Pete,Aberdeen,185,554
6,George,Aberdeen,189,944
7,John,Aberdeen,190,759
8,Terry,Aberdeen,190,759
9,Karl,Aberdeen,190,759


= «предыдущая группа ровесников + текущая группа + следующая группа»

| weight | Группа | Какие группы попадают | Какие строки | Сумма |
|--------|--------|------------------------|--------------|-------|
| 165    | G1     | G1, G2                 | 165, 175, 175 | 515  |
| 175    | G2     | G1, G2, G3             | 165, 175, 175, 180, 180 | 875 |
| 175    | G2     | G1, G2, G3             | 165, 175, 175, 180, 180 | 875 |
| 180    | G3     | G2, G3, G4             | 175, 175, 180, 180, 185 | 895 |
| 180    | G3     | G2, G3, G4             | 175, 175, 180, 180, 185 | 895 |
| 185    | G4     | G3, G4                 | 180, 180, 185 | 545  |

**Просто пример того, как сделать накопительную сумму исходя из всего вышесказанного:**

In [11]:
df = pd.read_sql(text("""SELECT first_name, birth_city, weight,
                                SUM(weight) OVER (PARTITION BY birth_city ORDER BY weight ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS total_weight
                        FROM players
                        WHERE birth_city IS NOT NULL AND weight IS NOT NULL
                            """), engine2)
df.head(10)

,first_name,birth_city,weight,total_weight
0,Otto,Aalen,180,180
1,Justin,Aberdeen,165,165
2,Guy,Aberdeen,175,340
3,Cal,Aberdeen,175,515
4,Tony,Aberdeen,180,695
5,Pete,Aberdeen,185,880
6,George,Aberdeen,189,1069
7,John,Aberdeen,190,1259
8,Terry,Aberdeen,190,1449
9,Karl,Aberdeen,190,1639


## Транзакции в SQL: BEGIN, COMMIT, ROLLBACK и ACID <a id="acid"></a>

**Транзакция** — это группа операций, которые выполняются как одно целое. Либо все операции применяются, либо ни одна. Промежуточных состояний не бывает

- **BEGIN** — начать транзакцию. С этого момента все операции накапливаются и пока не применяются к базе. Они как бы «в черновике»
- **COMMIT** — подтвердить. Все операции из черновика применяются к базе окончательно. После этого их уже не откатить
- **ROLLBACK** — откатить. Все операции из черновика отменяются. База возвращается к состоянию, которое было до BEGIN

Пример:
```sql
BEGIN;

UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

-- Если всё хорошо:
COMMIT;

-- Если что-то пошло не так:
ROLLBACK;
```

*В SQLAlchemy команда `with engine.begin() as conn:` под капотом автоматически выполняет BEGIN, COMMIT, ROLLBACK. То есть по сути всё что находится в блоке это одна транзакция*

**ACID** — это аббревиатура из четырёх гарантий, которые даёт транзакционная база данных. На уровне идеи — вот что это значит:

| Свойство | Что значит |
|---|---|
| **Atomicity (Атомарность)** | Либо оба счёта обновились, либо ни один. Не бывает «списал, но не зачислил». |
| **Consistency (Согласованность)** | Нельзя уйти в минус, если это запрещено правилами. |
| **Isolation (Изолированность)** | Пока перевод не завершён, другие клиенты не видят «промежуточный» баланс. |
| **Durability (Долговечность)** | После подтверждения перевода он не исчезнет, даже если банк загорится. |